# Apply Orbit Vector

Figuring out how to apply orbit vector files to Sentinel-1 Images.  This follows the procedure outlined in the [s1tbx code](https://github.com/senbox-org/s1tbx/blob/c10b60f535063db77bf1588dd25967731b61c69c/s1tbx-io-ephemeris/src/main/java/org/esa/s1tbx/orbits/gpf/ApplyOrbitFileOp.java#L369)

1. Get orbit file with valid time period and user specified orbit file type.
2. Get the old tie point grid of the source image: latitude, longitude, slant range time and incidence angle.
3. Repeat the following steps for each new tie point in the new tie point grid:
    1)  Get the range line index y for the tie point;
    2)  Get zero Doppler time t for the range line.
    3)  Compute satellite position and velocity for the zero Doppler time t using cubic interpolation. (dorisReader)
    4)  Get sample number x (index in the range line).
    5)  Get slant range time for pixel (x, y) from the old slant range time tie point grid.
    6)  Get incidence angle for pixel (x, y) from the old incidence angle tie point grid.
    7)  Get latitude for pixel (x, y) from the old latitude tie point grid.
    8)  Get longitude for pixel (x, y) from the old longitude tie point grid.
    9)  Convert (latitude, longitude, h = 0) to global Cartesian coordinate (x0, y0, z0).
    10) Solve Range equation, Doppler equation and Earth equation system for accurate (x, y, z) using Newton's method with (x0, y0, z0) as initial point.
    11) Convert (x, y, z) back to (latitude, longitude, h).
    12) Save the new latitude and longitude for current tie point.
4. Create new geocoding with the newly computed latitude and longitude tie points.
5. Update orbit state vectors in the metadata:
    1) Get zero Doppler time for each orbit state vector in the metadata of the source image.
    2) Compute new orbit state vector for the zero Doppler time using cubic interpolation.
    3) Save the new orbit state vector in the target product.

Product specification is given in https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-1-sar/product-types-processing-levels/level-1

See also Page 267 of Woodhouse.

In [70]:
import sys
sys.path.append('..')  #/heS1denoise')

import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from heS1denoise.sentinel1image import S1Image

In [71]:
DATAPATH = Path('/media', 'apbarret', 'andypbarrett_work', 'Data', 'Sentinel1_Sun')
IMAGEPATH = DATAPATH / 'S1A_EW_GRDM_1SDH_20180902T165032_20180902T165132_023522_028FAA_35D2.zip'

In [72]:
s1 = S1Image(IMAGEPATH)

10:24:46|40|nansat|_get_dataset_metadata|GDAL could not open /media/apbarret/andypbarrett_work/Data/Sentinel1_Sun/S1A_EW_GRDM_1SDH_20180902T165032_20180902T165132_023522_028FAA_35D2.zip, trying to read with Nansat mappers...


Geolocation tiepoints are accessed by `import_geolocationGridPoint`.  This returns a `dict` containing azimuthTime, slantRangeTime, line and pixel numbers, latitude, longitude, height (this is not 0), incidence angle and elevation angle.

```
['azimuthTime',
 'slantRangeTime',
 'line',
 'pixel',
 'latitude',
 'longitude',
 'height',
 'incidenceAngle',
 'elevationAngle']
```

In [73]:
geolocation_hh = s1.import_geolocationGridPoint('HH')

In [74]:
npoints = len(geolocation_hh['line'])
nline = len(set(geolocation_hh['line']))
npixel = npoints // nline
npoints, nline, npixel

(462, 22, 21)

## Open and read POEORB file

This is a zipped xml file obtained from https://scihub.copernicus.eu/gnss/#/home  There is an API to grab these.  For testing I will use a local file.

In [75]:
POEORB_PATH = Path('/home/apbarret/Data/Sentinel-1/POEORB/S1A/2020/12')
poeorb_file = POEORB_PATH / 'S1A_OPER_AUX_POEORB_OPOD_20210318T212943_V20201215T225942_20201217T005942.EOF.zip'

In [89]:
import zipfile
import xml.etree.ElementTree as ET
import datetime as dt

def load_orbit_vector_file(poeorb_zipfile):
    """Loads POEORB file
    
    :poeorb_zipfile: path to AUX_POEORB zipfile
    
    :returns: root of xmltree object
    """
    with zipfile.ZipFile(poeorb_zipfile) as zf:
        # Expects only one file
        nfile = len(zf.namelist())
        assert nfile == 1, f'Expects one file only, found {nfile}'
        
        fname = zf.namelist()[0]
        with zf.open(fname) as f:
            tree = ET.parse(f)
            
    return tree.getroot()

def parse_datetime(time_str):
    return dt.datetime.fromisoformat(time_str.split('=')[1])

def parse_osv_element(child):
    """Returns child tag and text as tuple"""
    key, value = child.tag, child.text
    if key in ['TAI', 'UTC', 'UT1']:
        value = parse_datetime(value)
    return (key, value)

def parse_osv(one_osv):
    """Parses a single OSV branch and returns a dict of elements"""
    return dict([parse_osv_element(child) for child in one_osv])
        
def get_list_of_osvs(list_of_osvs):
    """Parses the List_of_OSVs in the Data_Block sections
    :list_of_osvs: an ElementTree object containing List_of_OSVs
    
    :returns: list of dicts
    """
    orbit_state_vector_list = []
    for osv in list_of_osvs.findall('OSV'):
        orbit_state_vector_list.append(parse_osv(osv))
    return orbit_state_vector_list

class OrbitVector():
    """Class to hold Orbit Vector data"""
    
    def __init__(self, poeorb_zipfile):
        root = load_orbit_vector_file(poeorb_zipfile)
        
        header = root.find('Earth_Explorer_Header/Fixed_Header')
        self.file_name = header.find('File_Name').text
        self.file_description = header.find('File_Description').text
        self.mission = header.find('Mission').text
        self.file_class = header.find('File_Class').text
        self.file_type = header.find('File_Type').text
        self.validity_start = header.find('Validity_Period/Validity_Start').text
        self.validity_end = header.find('Validity_Period/Validity_Stop').text
        self.file_version = header.find('File_Version').text
        self.system = header.find('Source/System').text
        self.creator = header.find('Source/Creator').text
        self.creator_version = header.find('Source/Creator_Version').text
        self.creation_date = header.find('Source/Creation_Date').text
        
        list_of_osvs = root.find('Data_Block/List_of_OSVs')
        self.count = list_of_osvs.get('count')
        self.list_of_osvs = get_list_of_osvs(list_of_osvs)
        
        
    def __repr__(self):
        return ('Orbit State Vectors class\n'
                f'File Name: {self.file_name}\n'
                f'File Desription: {self.file_description}\n'
                f'Mission: {self.mission}\n'
                f'Valid from: {self.validity_start}\n'
                f'Valid to: {self.validity_start}\n'
                f'N-OSVs: {self.count}\n')

In [90]:
orbit_vector = OrbitVector(poeorb_file)

In [91]:
orbit_vector

Orbit State Vectors class
File Name: S1A_OPER_AUX_POEORB_OPOD_20210318T212943_V20201215T225942_20201217T005942
File Desription: Precise Orbit Ephemerides (POE) Orbit File
Mission: Sentinel-1A
Valid from: UTC=2020-12-15T22:59:42
Valid to: UTC=2020-12-15T22:59:42
N-OSVs: 9361

## Algorithm to update tie points

_Notes_

```
final int subSamplingX = sourceImageWidth / (targetTiePointGridWidth - 1);
final int subSamplingY = sourceImageHeight / (targetTiePointGridHeight - 1);
```

`sourceImageWidth` and `sourceImageHeight` are likely width and height of raster in meters

In [98]:
s1.shape()

(10116, 10399)